# Part 3: Exploratory Data Analysis with SQL

In [1]:
%load_ext sql

In [2]:
import csv
import sqlite3
import prettytable

In [4]:
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect('my_data1.db')
cur = con.cursor()

In [5]:
%sql sqlite:///my_data1.db

Connecting to 'sqlite:///my_data1.db'

In [6]:
import pandas as pd

df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv')
df.to_sql('SPACEXTBL', con, if_exists = 'replace', index = False, method = 'multi')



101

In [7]:
#drop the table if exists

%sql DROP TABLE IF EXISTS SPACEXTABLE;

Running query in 'sqlite:///my_data1.db'

++
||
++
++

In [8]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null


Running query in 'sqlite:///my_data1.db'

++
||
++
++

In [16]:
%sql SELECT * FROM SPACEXTABLE LIMIT 5

Running query in 'sqlite:///my_data1.db'

Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [30]:
%sql select DISTINCT(Landing_Outcome) from SPACEXTABLE LIMIT 20

Running query in 'sqlite:///my_data1.db'

Landing_Outcome
Failure (parachute)
No attempt
Uncontrolled (ocean)
Controlled (ocean)
Failure (drone ship)
Precluded (drone ship)
Success (ground pad)
Success (drone ship)
Success
Failure


## Task 1: Display the names of the unique launch sites in the space mission

In [17]:
%sql select DISTINCT(Launch_Site) from SPACEXTABLE LIMIT 20

Running query in 'sqlite:///my_data1.db'

Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


## Task 2: Display 5 records where launch sites Begin WITH string 'CCA'

In [21]:
%sql select * from SPACEXTABLE\
WHERE Launch_Site LIKE '%CCA%'\
LIMIT 5

Running query in 'sqlite:///my_data1.db'

Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


## Task 3: Display the total payload mass carried by boosters launched by NASA (CRS)

In [25]:
%sql SELECT PAYLOAD_MASS__KG_ FROM SPACEXTABLE\
WHERE Customer = 'NASA (CRS)'

Running query in 'sqlite:///my_data1.db'

PAYLOAD_MASS__KG_
500
677
2296
2216
2395
1898
1952
3136
2257
2490


## Task 4: Display Average payload mass carried by booster version F9

In [28]:
%sql SELECT AVG(PAYLOAD_MASS__KG_) FROM SPACEXTABLE\
WHERE Booster_Version LIKE '%F9%'

Running query in 'sqlite:///my_data1.db'

AVG(PAYLOAD_MASS__KG_)
6138.287128712871


## Task 5: List the date when the first succcesful landing outcome in ground pad was achieved

In [32]:
%sql SELECT Date FROM SPACEXTABLE\
WHERE Landing_Outcome = 'Success (ground pad)'\
ORDER BY Date ASC\
LIMIT 1

Running query in 'sqlite:///my_data1.db'

Date
2015-12-22


## Task 6: List the names of the boosters which have succes in drone ship and have payload mass greater than 4000 but less than 6000

In [33]:
%sql SELECT Booster_Version FROM SPACEXTABLE\
WHERE Landing_Outcome = 'Precluded (drone ship)'\
AND 4000 < PAYLOAD_MASS__KG_ <6000

Running query in 'sqlite:///my_data1.db'

Booster_Version
F9 v1.1 B1018


## Task 7: List the total number of successful and failure mission ]outcomes

In [46]:
%sql SELECT SUM(Mission_Outcome LIKE '%Success%') AS SUCCESS, SUM(Mission_Outcome LIKE '%Failure%') AS FAILURE\
FROM SPACEXTABLE

Running query in 'sqlite:///my_data1.db'

SUCCESS,FAILURE
100,1


## Task 8: List all the booster_version that have carried the mmaximum payload mass using a subquery with a suitable aggregate function

In [51]:
%sql SELECT Booster_Version FROM SPACEXTABLE\
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)

Running query in 'sqlite:///my_data1.db'

Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


## Task 9: List the record which display the month names, failure landing outcomes, in droneship, booster version, launchsite for months in year 2015

In [57]:
%sql SELECT  substr(Date, 6,2) AS Month, Booster_Version, Launch_Site, Landing_Outcome FROM SPACEXTABLE\
WHERE Landing_Outcome LIKE '%Failure%'\
AND substr(Date,0,5)='2015'

Running query in 'sqlite:///my_data1.db'

Month,Booster_Version,Launch_Site,Landing_Outcome
01,F9 v1.1 B1012,CCAFS LC-40,Failure (drone ship)
04,F9 v1.1 B1015,CCAFS LC-40,Failure (drone ship)


## Taks 10: Rank the count of landing outcomes (such as Failure (Drone Ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20 in descending order

In [60]:
%sql SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTABLE\
WHERE DATE BETWEEN '2010-06.04' AND '2017-03-20'\
GROUP BY Landing_Outcome\
ORDER BY Count DESC

Running query in 'sqlite:///my_data1.db'

Landing_Outcome,Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Precluded (drone ship),1
Failure (parachute),1


In [ ]:
cur.close()

In [ ]:
conn.close()